In [156]:
import pandas as pd
import json
from pprint import pprint as pp

# Setup

## Load data

In [157]:
years = [2022, 2023, 2024]
data = {}
for year in years:
    with open(f'data/fixtures/Prem/{year}.json', 'r') as f:
        data[year] = {'raw': json.load(f)['response']}

## Get data in same form as paper

- Tabular
- One row per match
- Columns: Home score, Away score, Home team, Away team

In [158]:
for year, d in data.items():
    data[year]['fixtures'] = []
    for fixture_info in d['raw']:
        data[year]['fixtures'].append({
            'id': fixture_info['fixture']['id'],
            'home': fixture_info['teams']['home']['name'],
            'home_goals': fixture_info['goals']['home'],
            'away': fixture_info['teams']['away']['name'],
            'away_goals': fixture_info['goals']['away'],
        })
    data[year]['df'] = pd.DataFrame(data[year]['fixtures'])

In [159]:
# SPlit years into training and testing
df_train = pd.concat([data[year]['df'] for year in years[:-1]])
df_test = data[years[-1]]['df']

# Filter to only 0-4 goals
df_train_4 = df_train[(df_train['home_goals'] < 5) & (df_train['away_goals'] < 5)]
df_test_4 = df_test[(df_test['home_goals'] < 5) & (df_test['away_goals'] < 5)]

# Exploring the data

### Empirical estimates of each score prob for joint and marginal prob functions

standard error for a proportion $p_i$ in n trials is $\sqrt{\frac{p_i (1 - p_i)}{n}}$

In [160]:
train_len = df_train.shape[0]

In [161]:
estimates = {'prob': {}, 'se': {}}
for ag in range(10):
    ag_df = df_train[df_train['away_goals'] == ag]
    estimates['prob'][ag] = {}
    estimates['se'][ag] = {}
    for hg in range(10):
        ag_hg_df = ag_df[ag_df['home_goals'] == hg]
        p = ag_hg_df.shape[0] / train_len
        estimates['prob'][ag][hg] = p
        estimates['se'][ag][hg] = (p * (1 - p) / train_len) ** 0.5

In [162]:
prob_df = pd.DataFrame(estimates['prob'])
se_df = pd.DataFrame(estimates['se'])

#### Table 1

In [ ]:
prob_df   # Columns are away goals, indices are home

,0,1,2,3,4,5,6,7,8,9
0,0.044737,0.057895,0.055263,0.021053,0.010526,0.003947,0.002632,0.0,0.001316,0.0
1,0.089474,0.100000,0.059211,0.032895,0.018421,0.005263,0.001316,0.0,0.000000,0.0
2,0.065789,0.085526,0.061842,0.021053,0.010526,0.001316,0.000000,0.0,0.000000,0.0
3,0.043421,0.056579,0.025000,0.011842,0.002632,0.000000,0.000000,0.0,0.000000,0.0
4,0.018421,0.025000,0.011842,0.011842,0.003947,0.000000,0.000000,0.0,0.000000,0.0
5,0.013158,0.007895,0.003947,0.001316,0.000000,0.000000,0.000000,0.0,0.000000,0.0
6,0.003947,0.003947,0.001316,0.001316,0.000000,0.000000,0.000000,0.0,0.000000,0.0
7,0.001316,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
9,0.001316,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0


In [173]:
cell_w = 12
empty_cell = ' '*cell_w
print(f'{empty_cell} {'Away':^{cell_w}} {0:^{cell_w}} {1:^{cell_w}} {2:^{cell_w}} {3:^{cell_w}} {4:^{cell_w}}')

# Away goals only row
print(f'{'Home':^{cell_w}} {empty_cell}', end=' ')
for ag in range(5):
    prob = prob_df.loc[:, ag].sum()
    se = se_df[ag].sum()
    cell = f'{round(100*prob, 1)} ({round(100*se, 2)})'
    print(f'{cell:^{cell_w}}', end=' ')
print()

# Main table contents
for hg in range(5):
    prob = prob_df.loc[hg, :].sum()
    se = se_df[hg].sum()
    cell = f'{round(100*prob, 1)} ({round(100*se, 2)})'
    print(f'{hg:^{cell_w}} {cell:^{cell_w}}', end=' ')
    for ag in range(5):
        prob = prob_df.loc[hg, ag]
        se = se_df.loc[hg, ag]
        cell = f'{round(100*prob, 1)} ({round(100*se, 2)})'
        print(f'{cell:^{cell_w}}', end=' ')
    print()

                 Away          0            1            2            3            4      
    Home                  28.2 (4.82)   33.7 (4.9)  21.8 (3.88)  10.1 (2.74)   4.6 (1.64)  
     0       19.7 (4.82)   4.5 (0.75)   5.8 (0.85)   5.5 (0.83)   2.1 (0.52)   1.1 (0.37)  
     1        30.7 (4.9)   8.9 (1.04)  10.0 (1.09)   5.9 (0.86)   3.3 (0.65)   1.8 (0.49)  
     2       24.6 (3.88)   6.6 (0.9)    8.6 (1.01)   6.2 (0.87)   2.1 (0.52)   1.1 (0.37)  
     3       13.9 (2.74)   4.3 (0.74)   5.7 (0.84)   2.5 (0.57)   1.2 (0.39)   0.3 (0.19)  
     4        7.1 (1.64)   1.8 (0.49)   2.5 (0.57)   1.2 (0.39)   1.2 (0.39)   0.4 (0.23)  


In [174]:
home, draw, away = 0, 0, 0
for ag in range(5):
    for hg in range(5):
        prob = prob_df.loc[hg, ag]
        if hg > ag:
            home += prob
        elif hg == ag:
            draw += prob
        else:
            away += prob
print(f'Home:Draw:Away = {round(100*home)}:{round(100*draw)}:{round(100*away)}')

Home:Draw:Away = 43:22:29


Comparing these results (Prem 2022/23 & 2023/24) to the results in the paper (EFL 1992/93 - 1994/95) the proportion of home wins is very similar, but away wins have risen and draws dropped.

## Examining data to check Poisson fit

In [176]:
prob_df.loc[:, 0]

0    0.044737
1    0.089474
2    0.065789
3    0.043421
4    0.018421
5    0.013158
6    0.003947
7    0.001316
8    0.000000
9    0.001316
Name: 0, dtype: float64

In [ ]:
joint_product_ratios = {}
# Used the same goal numbers as paper for easy comparison, above these are rare anyway
for ag in range(6):
    joint_product_ratios[ag] = {}
    marg_a = prob_df.loc[:, ag].sum()
    for hg in range(7):
        marg_h = prob_df.loc[hg, :].sum()
        joint = prob_df.loc[hg, ag]
        ratio = joint / (marg_a * marg_h)
        joint_product_ratios[ag][hg] = ratio*100

In [181]:
# !!! ADD BOOTSTRAP ERRORS

In [179]:
joint_df = pd.DataFrame(joint_product_ratios)

In [ ]:
joint_df   # 0s represent no scoreline existing

,0,1,2,3,4,5
0,80.498442,87.083333,128.192771,105.281385,115.809524,190.000000
1,103.646063,96.834764,88.422359,105.902681,130.472103,163.090129
2,94.957269,103.191845,115.069905,84.450309,92.895340,50.802139
3,110.562511,120.430425,82.064105,83.802989,40.970350,0.000000
4,92.073382,104.456019,76.305221,164.502165,120.634921,0.000000
5,177.570093,89.062500,68.674699,49.350649,0.000000,0.000000
6,133.177570,111.328125,57.228916,123.376623,0.000000,0.000000


Quite a few not close to 100, possibly due to much lower sample size than paper (380 compared to 6629) currently.

# Modelling